In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/flight-delays-dataset/flight_delays.csv


# **Importing Necessary Packages**

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load the data

In [8]:
df = pd.read_csv('/kaggle/input/flight-delays-dataset/flight_delays.csv')

 # Convert date and time columns to datetime and Extracting useful features from datetime

In [9]:
date_columns = ['ScheduledDeparture', 'ActualDeparture', 'ScheduledArrival', 'ActualArrival']
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')  # Handle any parsing issues


df['DayOfWeek'] = df['ScheduledDeparture'].dt.dayofweek
df['Month'] = df['ScheduledDeparture'].dt.month
df['Hour'] = df['ScheduledDeparture'].dt.hour


In [10]:
df.head()

,FlightID,Airline,FlightNumber,Origin,Destination,ScheduledDeparture,ActualDeparture,ScheduledArrival,ActualArrival,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,TailNumber,Distance,DayOfWeek,Month,Hour
0,1,United,4558,ORD,MIA,2024-09-01 08:11:00,2024-09-01 08:30:00,2024-09-01 12:11:00,2024-09-01 12:19:00,8,Weather,True,False,Boeing 737,N71066,1031,6,9,8
1,2,Delta,8021,LAX,MIA,2024-09-01 10:25:00,2024-09-01 10:41:00,2024-09-01 13:25:00,2024-09-01 13:27:00,2,Air Traffic Control,True,True,Airbus A320,N22657,1006,6,9,10
2,3,Southwest,7520,DFW,SFO,2024-09-01 16:53:00,2024-09-01 17:05:00,2024-09-01 17:53:00,2024-09-01 18:07:00,14,Weather,True,True,Boeing 737,N95611,2980,6,9,16
3,4,Delta,2046,ORD,BOS,2024-09-01 14:44:00,2024-09-01 15:04:00,2024-09-01 18:44:00,2024-09-01 18:34:00,-10,NaN,False,False,Boeing 777,N90029,1408,6,9,14
4,5,Delta,6049,LAX,SEA,2024-09-01 01:51:00,2024-09-01 02:08:00,2024-09-01 05:51:00,2024-09-01 06:15:00,24,Air Traffic Control,False,True,Boeing 737,N27417,2298,6,9,1


# Dropping unnecessary columns (keep only useful ones) and Handling missing values in DelayMinutes

In [11]:
columns_to_drop = ['ScheduledDeparture', 'ActualDeparture', 
                   'ScheduledArrival', 'ActualArrival', 'TailNumber']
df = df.drop(columns=columns_to_drop)

df['DelayMinutes'] = pd.to_numeric(df['DelayMinutes'], errors='coerce').fillna(0)

In [12]:
df.head()

,FlightID,Airline,FlightNumber,Origin,Destination,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,Distance,DayOfWeek,Month,Hour
0,1,United,4558,ORD,MIA,8,Weather,True,False,Boeing 737,1031,6,9,8
1,2,Delta,8021,LAX,MIA,2,Air Traffic Control,True,True,Airbus A320,1006,6,9,10
2,3,Southwest,7520,DFW,SFO,14,Weather,True,True,Boeing 737,2980,6,9,16
3,4,Delta,2046,ORD,BOS,-10,NaN,False,False,Boeing 777,1408,6,9,14
4,5,Delta,6049,LAX,SEA,24,Air Traffic Control,False,True,Boeing 737,2298,6,9,1


# Convert boolean columns to int and defining features and target

In [13]:

bool_columns = ['Cancelled', 'Diverted']
for col in bool_columns:
    df[col] = df[col].astype(int)


features = ['Airline', 'Origin', 'Destination', 'DayOfWeek', 'Month', 'Hour', 
            'Cancelled', 'Diverted', 'AircraftType', 'Distance']
target = 'DelayMinutes'

X = df[features]
y = df[target]


In [14]:
df

,FlightID,Airline,FlightNumber,Origin,Destination,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,Distance,DayOfWeek,Month,Hour
0,1,United,4558,ORD,MIA,8,Weather,1,0,Boeing 737,1031,6,9,8
1,2,Delta,8021,LAX,MIA,2,Air Traffic Control,1,1,Airbus A320,1006,6,9,10
2,3,Southwest,7520,DFW,SFO,14,Weather,1,1,Boeing 737,2980,6,9,16
3,4,Delta,2046,ORD,BOS,-10,NaN,0,0,Boeing 777,1408,6,9,14
4,5,Delta,6049,LAX,SEA,24,Air Traffic Control,0,1,Boeing 737,2298,6,9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1747622,1747623,United,4145,DFW,MIA,-5,NaN,0,1,Boeing 737,2396,6,9,12
1747623,1747624,United,2155,ATL,SEA,20,Weather,1,0,Boeing 777,2185,6,9,20
1747624,1747625,Delta,4878,JFK,SFO,-6,NaN,0,0,Boeing 777,361,6,9,3
1747625,1747626,Delta,2940,JFK,SEA,16,Maintenance,0,1,Airbus A320,2793,6,9,8


In [15]:
total_nan_values = df['DelayReason'].isnull().sum()

print("Total NaN values in the column:", total_nan_values)

Total NaN values in the column: 468873


# Define features and target

In [16]:

features = ['Airline', 'Origin', 'Destination', 'DayOfWeek', 'Month', 'Hour', 
            'Cancelled', 'Diverted', 'AircraftType', 'Distance']
target = 'DelayMinutes'

X = df[features]
y = df[target]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define preprocessing steps


In [17]:
numeric_features = ['Distance', 'DayOfWeek', 'Month', 'Hour', 'Cancelled', 'Diverted']
categorical_features = ['Airline', 'Origin', 'Destination', 'AircraftType']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create a pipeline with preprocessor and model and fit the model

In [19]:
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Distance', 'DayOfWeek',
                                                   'Month', 'Hour', 'Cancelled',
                                                   'Diverted']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Airline', 'Origin',
                                                   'Destination',
                                                   'AircraftType'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

# Make predictions

In [20]:
y_pred = model.predict(X_test)

# Evaluate the model

In [21]:

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

# Function to predict delay for new data
def predict_delay(new_data):
    return model.predict(new_data)


Mean Squared Error: 152.0112566321983
R-squared Score: -0.08518536280782762


In [34]:
# Example usage
new_flight = pd.DataFrame({
    'Airline': ['Delta'],
    'Origin': ['LAX'],
    'Destination': ['JFK'],
    'DayOfWeek': [4],
    'Month': [9],
    'Hour': [13],
    'Cancelled': [1],
    'Diverted': [0],
    'AircraftType': ['Boeing 737'],
    'Distance': [2475]
})

predicted_delay = predict_delay(new_flight)
print(f"Predicted delay: {predicted_delay[0]:.2f} minutes")

Predicted delay: 17.35 minutes


In [24]:
df

,FlightID,Airline,FlightNumber,Origin,Destination,DelayMinutes,DelayReason,Cancelled,Diverted,AircraftType,Distance,DayOfWeek,Month,Hour
0,1,United,4558,ORD,MIA,8,Weather,1,0,Boeing 737,1031,6,9,8
1,2,Delta,8021,LAX,MIA,2,Air Traffic Control,1,1,Airbus A320,1006,6,9,10
2,3,Southwest,7520,DFW,SFO,14,Weather,1,1,Boeing 737,2980,6,9,16
3,4,Delta,2046,ORD,BOS,-10,NaN,0,0,Boeing 777,1408,6,9,14
4,5,Delta,6049,LAX,SEA,24,Air Traffic Control,0,1,Boeing 737,2298,6,9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1747622,1747623,United,4145,DFW,MIA,-5,NaN,0,1,Boeing 737,2396,6,9,12
1747623,1747624,United,2155,ATL,SEA,20,Weather,1,0,Boeing 777,2185,6,9,20
1747624,1747625,Delta,4878,JFK,SFO,-6,NaN,0,0,Boeing 777,361,6,9,3
1747625,1747626,Delta,2940,JFK,SEA,16,Maintenance,0,1,Airbus A320,2793,6,9,8
